# သင်ခန်းစာ ၀၈ - မူလတည်းဖြင့် အမြောက်အမြားတိုက်ခိုက်သူ ဒီဇိုင်းပုံစံ


## ပြင်ဆင်ခြင်း


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ဘာကြောင့် Multi-Agent Systems လဲ?

ခရီးစဉ်အစီအစဉ်ကဲ့သို့သော လက်တွေ့ကိစ္စများတွင် ပညာရှင်အမျိုးမျိုးစွာ ပါဝင်ရသည် — သယ်ယူပို့ဆောင်ရေး၊ ဒေသဆိုင်ရာ သိပ္ပံ၊ ဘတ်ဂျက်သတ်မှတ်ခြင်းနှင့် အခြားများ။ တစ်ဦးတည်းသော အေးဂျင့်သည် အားလုံးကို ကိုင်တွယ်ရန် ကြိုးစားသည်မှာ တော်တော်လေး စီမံခန့်ခွဲရန်ခက်ခဲသွားပါတယ်။

Multi-agent systems များသည် **အထူးပြုမှု** မှတဆင့် ဒီပြဿနာကို ဖြေရှင်းပေးသည် - တစ်ဦးချင်းစီပြောရမယ်ဆိုရင် တစ်ခုတည်းသော အရေးပါတဲ့ နယ်ပယ်တစ်ခုကိုသာ ဦးတည်၍ ပိုမိုမြင့်မားသော အရည်အသွေးရရှိစေသည်။ ထိုအပြင် **မျှဝေရေးပမာဏတိုးတက်မှု** ကိုလည်း တိုးတက်စေသည် — အသစ်သော အေးဂျင့်များ (ဥပမာ၊ လေယာဉ်အထူးပြုနယ်ပယ်၊ စားသောက်ဆိုင်သုံးသပ်သူ) ကို လက်ရှိလုပ်ငန်းစဉ်ကို ပြန်ရေးမလို့မလိုဘဲ ထပ်တိုးထည့်နိုင်သည်။ အေးဂျင့်များကို ချိတ်ဆက်သည့် စနစ်တကျဖြစ်သော လမ်းကြောင်းတစ်ခုမှတဆင့် ကိုယ်ရေးရာစဉ်အားတစ်ဦးမှတစ်ဦးသို့ ပေးပို့ပြီး တွဲဖက်ဆောင်ရွက်သည်။


## အထူးပြုထားသော ကိုယ်စားလှယ်များ ဖန်တီးခြင်း


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## တစ်လိုက်လုပ်ဆောင်မှု Workflow တည်ဆောက်ခြင်း

`WorkflowBuilder` သည် ကိုယ်စားလှယ်များအား အညွှန်းထားသော ဇယားတစ်ခုအဖြစ် ချိတ်ဆက်ပေးနိုင်စေသည်။ ဒီမှာ ကျွန်ုပ်တို့သည် အဆင့်၂ဆင့် Pipeline ရိုးရှင်းတစ်ခု ဖန်တီးမည်ဖြစ်သည် - **TravelPlanner** သည် ခရီးအစီအစဉ်ကို ရေးဆွဲပြီး၊ ထို့နောက် **TravelConcierge** သည် ဤအစီအစဉ်ကို လေ့လာပြီး တိုးတက်အောင် ပြုလုပ်သည်။


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## အလုပ်စဉ်ထဲမှာ ပိုမိုအေးဂျင့်တွေ ဖြည့်တိုးခြင်း

multi-agent ပုံစံရဲ့ အကြီးမားဆုံးအားသာချက်တွေထဲက တစ်ခုကတော့ တိုးချဲ့ဖို့လွယ်ကူမှုပါ။ အောက်မှာ ဧည့်သပိတ်၏ ငွေကြေးအစီအစဉ်နဲ့ပြောင်းဆိုမှုများကို စစ်ဆေးပြီး၊ ဘတ်ဂျက်ကျော်လွန်နိုင်မယ့် အရာတွေကို အမှတ်အသား ပေးတဲ့ **BudgetReviewer** အေးဂျင့်ကို ဖြည့်တိုးထားပါတယ်။ လက်ရှိ workflow က အေးဂျင့်သုံးယောက်ကို အဆင့်လိုက် ပြေးဆဲဖြစ်ပါတယ်။

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## အနှစ်ချုပ်

ဒီသင်ခန်းစာမှာ သင်လေ့လာခဲ့တာတွေကတော့ -

1. **အထူးပြု အေးဂျင့်တွေ ဖန်တီးခြင်း** — တစ်ဦးချင်းစီမှာ ရည်ရွယ်ချက်ချထားသော အခန်းကဏ္ဍ (စီမံကိန်းရေးဆွဲခြင်း၊ ဆက်သွယ်ရေး၊ ဘတ်ဂျက် ပြန်လည်သုံးသပ်ခြင်း) ပါဝင်သည်။
2. **Agent တွေကို `WorkflowBuilder` နဲ့ `add_edge` ကို အသုံးပြုကာ အဆက်အသွယ်စဉ်ကျသော အလုပ်လမ်းကြောင်းထဲတစ်ခုချိတ်ဆက်ခြင်း**။
3. **အဆင့်မြင့် multi-agent လုပ်ငန်းစဉ်မှ ထွက်ရှိမှုများကို စတီးမြှင့်ပြီး မည်သူက ပြောဆိုနေသည်ကို ချိန်ဆန့်ခြင်း**။
4. **အလုပ်လမ်းကြောင်းကို ကျန်ရှိသေးသော agent များကို ပြင်ဆင်ခြင်းမရှိဘဲ အသစ်ထည့်သွင်း၍ တိုးချဲ့နိုင်ခြင်း**။

Multi-agent ဒီဇိုင်းပုံစံက agent တစ်ဦးချင်းစီကို ရိုးရှင်းစွာထားပြီး တစ်ဦးတည်း agent ထက် ပိုပြီး သေချာစွာ ပြန်လည်သုံးသပ်ထားသော ရလဒ်များ ပိုပေါ်ရှိစေနိုင်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
